In [12]:
import json
import re
from pathlib import Path
from typing import Optional

INCIDENTS_PATH = Path('1_src_incidents.json')
LABS_ROOT = Path('DeFiHackLabs')
OUTPUT_PATH = Path('2_ethereum_incidents_with_attack_txs.json')
ETHERSCAN_TX_PREFIX = 'https://etherscan.io/tx/'

TX_HASH_RE = re.compile(r"0x[a-fA-F0-9]{64}")


def extract_attack_tx(contract_path: Optional[str]) -> Optional[str]:
    """Return first attack tx hash mentioned in the corresponding DeFiHackLabs file."""
    if not contract_path:
        return None
    file_path = LABS_ROOT / contract_path
    if not file_path.exists():
        return None
    text = file_path.read_text(encoding='utf-8', errors='ignore')
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if 'tx' not in line.lower():
            continue
        match = TX_HASH_RE.search(line)
        if match:
            tx_hash = match.group(0).lower()
            return f"{ETHERSCAN_TX_PREFIX}{tx_hash}"
    return None


def main() -> None:
    if not LABS_ROOT.exists():
        raise SystemExit('DeFiHackLabs repo is required to extract attack transactions.')

    incidents = json.loads(INCIDENTS_PATH.read_text())
    eth_incidents = [item for item in incidents if str(item.get('chain')) == 'Ethereum']
    eth_incidents.sort(key=lambda item: item.get('date', ''), reverse=True)

    enriched: list[dict] = []
    missing: list[dict] = []

    for idx, incident in enumerate(eth_incidents, start=1):
        item = dict(incident)
        item['id'] = f'ETH-{idx:03d}'
        attack_tx = extract_attack_tx(item.get('Contract'))
        item['attack_tx'] = attack_tx
        if not attack_tx:
            missing.append(item)
        enriched.append(item)

    OUTPUT_PATH.write_text(json.dumps(enriched, indent=2), encoding='utf-8')
    print(f"{len(enriched)} Ethereum incidents written to {OUTPUT_PATH}")
    if missing:
        print(f"Missing attack tx for {len(missing)} incidents. Examples:")
        for sample in missing[:10]:
            print(f" - {sample.get('name')} ({sample.get('Contract')})")


if __name__ == '__main__':
    main()



249 Ethereum incidents written to 2_ethereum_incidents_with_attack_txs.json
Missing attack tx for 36 incidents. Examples:
 - EGGX (src/test/2024-02/EGGX_exp.sol)
 - LQDX (src/test/2024-01/LQDX_alert_exp.sol)
 - NOON (NO) (src/test/2023-05/NOON_exp.sol)
 - Silo finance (src/test/2023-04/silo_finance_exp.sol)
 - Swapos V2 (src/test/2023-04/Swapos_exp.sol)
 - MEVBOT_0xbaDc0dE (src/test/2022-09/MEVbadc0de_exp.sol)
 - Omni NFT (src/test/2022-07/Omni_exp.sol)
 - FlippazOne NFT (src/test/2022-07/FlippazOne_exp.sol)
 - Quixotic (src/test/2022-07/Quixotic_exp.sol)
 - SNOOD (src/test/2022-06/Snood_exp.sol)


In [15]:

with open('2_ethereum_incidents_with_attack_txs.json', 'r') as f:
    data = json.load(f)

data = [item for item in data if item.get('attack_tx')]

# print(f'Removed {count} items without attack tx')

with open('3_ethereum_incidents_with_only_attack_txs.json', 'w') as f:
    json.dump(data, f, indent=2)

print(f'Wrote {len(data)} items with attack tx to ethereum_incidents_with_only_attack_txs.json')

Wrote 213 items with attack tx to ethereum_incidents_with_only_attack_txs.json


In [22]:

import json
import os
import random
from pathlib import Path
from typing import Optional

from web3 import Web3

import dotenv
dotenv.load_dotenv()


INCIDENTS_WITH_ATTACKS = Path("3_ethereum_incidents_with_only_attack_txs.json")
BALANCED_OUTPUT_PATH = Path("4_ethereum_incidents_with_attack_and_normal_txs.json")
NORMAL_TXS_OUTPUT_PATH = Path("ethereum_normal_txs_by_incident.json")
NODE_URL = os.environ.get("NODE_URL")
if not NODE_URL:
    raise SystemExit("NODE_URL env var is required to fetch blocks for normal transaction sampling.")

w3 = Web3(Web3.HTTPProvider(NODE_URL))
random_seed = int(os.environ.get("NORMAL_TX_RANDOM_SEED", "2601"))
rng_stream = random.Random(random_seed)


def extract_hash_from_entry(entry: Optional[str]) -> Optional[str]:
    if not entry:
        return None
    value = entry.strip()
    if value.startswith("0x") and len(value) == 66:
        return value.lower()
    if "/tx/" in value:
        part = value.split("/tx/", 1)[1]
        part = part.split("?", 1)[0].split("/", 1)[0]
        if part.startswith("0x"):
            return part.lower()
    return None


def normalize_tx_hash(raw_hash) -> Optional[str]:
    if raw_hash is None:
        return None
    try:
        return Web3.to_hex(raw_hash).lower()
    except Exception:
        if isinstance(raw_hash, str):
            return raw_hash.lower()
    return None


def get_tx_attr(tx_obj, key: str):
    if isinstance(tx_obj, dict):
        return tx_obj.get(key)
    return getattr(tx_obj, key, None)


def normalize_address(addr: Optional[str]) -> Optional[str]:
    if not addr:
        return None
    try:
        return Web3.to_checksum_address(addr)
    except Exception:
        try:
            return Web3.to_checksum_address(Web3.to_hex(addr))
        except Exception:
            return addr


def choose_candidate(candidates, incident_seed: str):
    seed_value = f"{incident_seed}-{rng_stream.randint(0, 10**9)}"
    rng = random.Random(seed_value)
    return rng.choice(candidates)


records = json.loads(INCIDENTS_WITH_ATTACKS.read_text())
balanced_records: list[dict] = []
normal_summaries: list[dict] = []
skipped: list[tuple[str, str]] = []

for record in records:
    incident_id = record.get("id")
    attack_hash = extract_hash_from_entry(record.get("attack_tx"))
    if not attack_hash:
        skipped.append((incident_id, "missing attack tx hash"))
        balanced_records.append(record)
        continue

    try:
        tx = w3.eth.get_transaction(attack_hash)
    except Exception as exc:
        skipped.append((incident_id, f"could not load attack tx: {exc}"))
        balanced_records.append(record)
        continue

    block_number = get_tx_attr(tx, "blockNumber")
    if block_number is None:
        skipped.append((incident_id, "attack tx missing block number"))
        balanced_records.append(record)
        continue

    try:
        block = w3.eth.get_block(block_number, full_transactions=True)
    except Exception as exc:
        skipped.append((incident_id, f"could not load block {block_number}: {exc}"))
        balanced_records.append(record)
        continue

    candidates = []
    for tx_obj in block.transactions:
        tx_hash = normalize_tx_hash(get_tx_attr(tx_obj, "hash"))
        if not tx_hash or tx_hash == attack_hash:
            continue
        to_addr = normalize_address(get_tx_attr(tx_obj, "to"))
        if not to_addr:
            continue
        try:
            code = w3.eth.get_code(to_addr, block_identifier=block_number)
        except Exception:
            continue
        if not code:
            continue
        candidates.append((tx_hash, tx_obj, to_addr))

    if not candidates:
        skipped.append((incident_id, f"no contract tx found in block {block_number}"))
        balanced_records.append(record)
        continue

    chosen_hash, chosen_tx, chosen_to = choose_candidate(candidates, str(incident_id))
    normal_url = f"{ETHERSCAN_TX_PREFIX}{chosen_hash}"
    print(f"Incident {incident_id}: selected normal tx {normal_url} in block {block_number}")

    existing_normals = list(record.get("normal_txs") or [])
    record["normal_txs"] = [normal_url] + [tx for tx in existing_normals if tx != normal_url]

    normal_summaries.append(
        {
            "id": incident_id,
            "attack_tx": record.get("attack_tx"),
            "normal_tx": normal_url,
            "block_number": block_number,
            "normal_to": chosen_to,
            "normal_tx_index": get_tx_attr(chosen_tx, "transactionIndex"),
        }
    )
    balanced_records.append(record)

BALANCED_OUTPUT_PATH.write_text(json.dumps(balanced_records, indent=2), encoding="utf-8")
NORMAL_TXS_OUTPUT_PATH.write_text(json.dumps(normal_summaries, indent=2), encoding="utf-8")

print(f"Wrote {len(normal_summaries)} records with normal tx to {BALANCED_OUTPUT_PATH}")
print(f"Normal transaction links saved to {NORMAL_TXS_OUTPUT_PATH}")
if skipped:
    print(f"Skipped {len(skipped)} incidents due to errors")
    for incident_id, reason in skipped[:10]:
        print(f" - {incident_id}: {reason}")


Incident ETH-001: selected normal tx https://etherscan.io/tx/0xf38e0f7e3c6b74d41fe97ab148d6eab357d5a9bb0981f4cff6ff7541e7e3a0a8 in block 23914086
Incident ETH-002: selected normal tx https://etherscan.io/tx/0x1af7d5bfd46f094444e8a8e9177758f1f9779fbc69c47fc6c5ee5a010412a9f6 in block 23769387
Incident ETH-003: selected normal tx https://etherscan.io/tx/0x5a2a9f4debb01f38ccd2a10221f7ff3906f2955d8a7b8f7948918fc18dfd2cd7 in block 23717397
Incident ETH-004: selected normal tx https://etherscan.io/tx/0x23aeb223c181f5734cf840f3fd8baa0eea8e565f0be822b12ce78abb0a176e9e in block 23260641
Incident ETH-005: selected normal tx https://etherscan.io/tx/0xfb8da23125b7641d1240483f50116563f803900e90d2cbf75dceedd3ec90fc7a in block 23232613
Incident ETH-006: selected normal tx https://etherscan.io/tx/0x06533f5743d62747f6b0d32fc03abbca97e23e23e3ca96507fd6a5f268cab7fa in block 23145764
Incident ETH-007: selected normal tx https://etherscan.io/tx/0x8ca4737e48b8ca7bca38c90e4d14f8ab562cb55e366198108eac16b240e2c